# ***PHASE1: Compare Dataset***

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
import time

# ============================================================
# NOTE: CONFIG
# ============================================================

TRAIN_PRED_CSV = r""
TEST_PRED_CSV = r""

# ============================================================
# BASIC FUNCTIONS
# ============================================================
MAX_SCORE = 25
TRAIN_GT_CSV = "dr_plant_train.csv"
TEST_GT_CSV = "dr_plant_test.csv"

KEY_COL = "timestamp"

def is_missing(value):
    if pd.isna(value):
        return True

    value_str = str(value).strip().lower()
    return value_str in ["", "nan", "none", "null"]

def cells_equal(a, b, numeric_tol=1e-9, case_sensitive=False):
    if is_missing(a) and is_missing(b):
        return True

    if is_missing(a) or is_missing(b):
        return False

    try:
        return abs(float(a) - float(b)) <= numeric_tol
    except Exception:
        pass

    a_text = str(a).strip()
    b_text = str(b).strip()

    if not case_sensitive:
        a_text = a_text.lower()
        b_text = b_text.lower()

    return a_text == b_text

def load_table(file_path, sheet_name=None):
    file_path = Path(file_path)

    if file_path.suffix.lower() == ".csv":
        return pd.read_csv(file_path)

    if file_path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(file_path, sheet_name=sheet_name)

    raise ValueError("Only .csv, .xlsx, and .xls files are supported.")

# ============================================================
# COMPARISON FUNCTION
# ============================================================
def compare_tables(
    ground_truth_df,
    prediction_df,
    key_col=None,
    exclude_cols=None,
    max_score=25,
    numeric_tol=1e-9,
):
    gt = ground_truth_df.copy()
    pred = prediction_df.copy()

    gt.columns = [str(c).strip() for c in gt.columns]
    pred.columns = [str(c).strip() for c in pred.columns]

    exclude_cols = exclude_cols or []

    if key_col is not None:
        if key_col not in gt.columns:
            raise ValueError(f"key_col '{key_col}' not found in ground truth table.")

        if key_col not in pred.columns:
            raise ValueError(f"key_col '{key_col}' not found in prediction table.")

        if gt[key_col].duplicated().any():
            raise ValueError(f"Duplicate key values found in ground truth column '{key_col}'.")

        if pred[key_col].duplicated().any():
            raise ValueError(f"Duplicate key values found in prediction column '{key_col}'.")

        gt = gt.set_index(key_col, drop=False)
        pred = pred.set_index(key_col, drop=False)

        row_keys = sorted(set(gt.index).union(set(pred.index)))
        ignore_cols = set([key_col] + exclude_cols)

    else:
        gt["_row_no"] = range(1, len(gt) + 1)
        pred["_row_no"] = range(1, len(pred) + 1)

        gt = gt.set_index("_row_no")
        pred = pred.set_index("_row_no")

        row_keys = sorted(set(gt.index).union(set(pred.index)))
        ignore_cols = set(["_row_no"] + exclude_cols)

    compare_cols = sorted((set(gt.columns).union(set(pred.columns))) - ignore_cols)

    details = []

    for row_key in row_keys:
        for col in compare_cols:
            gt_row_exists = row_key in gt.index
            pred_row_exists = row_key in pred.index
            gt_col_exists = col in gt.columns
            pred_col_exists = col in pred.columns

            gt_exists = gt_row_exists and gt_col_exists
            pred_exists = pred_row_exists and pred_col_exists

            gt_value = gt.loc[row_key, col] if gt_exists else None
            pred_value = pred.loc[row_key, col] if pred_exists else None

            is_match = False

            if gt_exists and pred_exists:
                is_match = cells_equal(gt_value, pred_value, numeric_tol=numeric_tol)

            details.append({
                "row_key": row_key,
                "column": col,
                "ground_truth_value": gt_value,
                "prediction_value": pred_value,
                "match": is_match,
                "gt_exists": gt_exists,
                "prediction_exists": pred_exists,
            })

    detail_df = pd.DataFrame(details)

    total_cells = len(detail_df)
    matched_cells = int(detail_df["match"].sum())

    accuracy = matched_cells / total_cells if total_cells > 0 else 0
    score = accuracy * max_score

    summary = {
        "matched_cells": matched_cells,
        "total_cells": total_cells,
        "accuracy": round(accuracy, 4),
        "score_out_of_25": round(score, 4),
        "max_score": max_score,
    }

    row_score_df = (
        detail_df
        .groupby("row_key")
        .agg(
            matched_cells=("match", "sum"),
            total_cells=("match", "count")
        )
        .reset_index()
    )

    row_score_df["row_score_out_of_25"] = (
        row_score_df["matched_cells"] / row_score_df["total_cells"] * max_score
    ).round(4)

    return summary, detail_df, row_score_df

# ============================================================
# COMPARE ONE PAIR
# ============================================================
def compare_csv_pair(
    pair_name,
    ground_truth_csv,
    prediction_csv,
    key_col=None,
    max_score=25,
):
    gt_df = load_table(ground_truth_csv)
    pred_df = load_table(prediction_csv)

    summary, detail_df, row_score_df = compare_tables(
        ground_truth_df=gt_df,
        prediction_df=pred_df,
        key_col=key_col,
        max_score=max_score,
    )

    summary["pair_name"] = pair_name
    summary["ground_truth_file"] = str(ground_truth_csv)
    summary["prediction_file"] = str(prediction_csv)

    return summary, detail_df, row_score_df

# ============================================================
# MAIN RUN
# ============================================================

start_time = time.perf_counter()

train_summary, train_detail_df, train_row_score_df = compare_csv_pair(
    pair_name="train",
    ground_truth_csv=TRAIN_GT_CSV,
    prediction_csv=TRAIN_PRED_CSV,
    key_col=KEY_COL,
    max_score=MAX_SCORE,
)

test_summary, test_detail_df, test_row_score_df = compare_csv_pair(
    pair_name="test",
    ground_truth_csv=TEST_GT_CSV,
    prediction_csv=TEST_PRED_CSV,
    key_col=KEY_COL,
    max_score=MAX_SCORE,
)

end_time = time.perf_counter()
runtime_seconds = end_time - start_time

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ============================================================
# PRINT RESULT
# ============================================================

print("\n********** Final Summary Score **********")
print(f"Timestamp:      {now}")
print(f"Runtime:        {runtime_seconds:.4f} seconds")

print("\n========== TRAIN RESULT ==========")
print(f"Matched Cells:      {train_summary['matched_cells']}")
print(f"Total Cells:        {train_summary['total_cells']}")
print(f"Accuracy:           {train_summary['accuracy']:.4f}")
print(f"Score out of 25:    {train_summary['score_out_of_25']}")
print(f"Max Score:          {train_summary['max_score']}")

print("\n========== TEST RESULT ==========")
print(f"Matched Cells:      {test_summary['matched_cells']}")
print(f"Total Cells:        {test_summary['total_cells']}")
print(f"Accuracy:           {test_summary['accuracy']:.4f}")
print(f"Score out of 25:    {test_summary['score_out_of_25']}")
print(f"Max Score:          {test_summary['max_score']}")

print("\n*****************************************")
print(f"PHASE1:Total Score:        {(train_summary['score_out_of_25']+test_summary['score_out_of_25'])/2}")

# ***PHASE2: Calculate Acc of model***

In [ ]:
import json
import hashlib
from pathlib import Path
from datetime import datetime
import time


# ============================================================
# NOTE: CONFIG
# ============================================================
METADATA_JSON = r""

# ============================================================
# COLOR PRINT
# ============================================================
MAX_SCORE = 25
GREEN = "\033[92m"
RED = "\033[91m"
YELLOW = "\033[93m"
RESET = "\033[0m"
BASELINE_HASH_FILE = "file_integrity_baseline.json"

def green_pass(text):
    return f"{GREEN}PASSED{RESET} - {text}"

def red_fail(text):
    return f"{RED}FAILED{RESET} - {text}"

def yellow_warn(text):
    return f"{YELLOW}WARNING{RESET} - {text}"

# ============================================================
# FILE INTEGRITY CHECK
# ============================================================
def calculate_file_hash(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

def check_file_not_modified(file_path, baseline_hash_file):

    file_path = Path(file_path)
    baseline_hash_file = Path(baseline_hash_file)

    current_hash = calculate_file_hash(file_path)

    if not baseline_hash_file.exists():
        baseline_data = {
            "file_name": file_path.name,
            "baseline_hash": current_hash,
            "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        with open(baseline_hash_file, "w", encoding="utf-8") as f:
            json.dump(baseline_data, f, indent=4)

        return {
            "status": True,
            "message": green_pass("file not modified, baseline hash created"),
            "current_hash": current_hash,
            "baseline_hash": current_hash,
            "baseline_created": True
        }


    with open(baseline_hash_file, "r", encoding="utf-8") as f:
        baseline_data = json.load(f)

    baseline_hash = baseline_data.get("baseline_hash")

    if current_hash == baseline_hash:
        return {
            "status": True,
            "message": green_pass("file not modified"),
            "current_hash": current_hash,
            "baseline_hash": baseline_hash,
            "baseline_created": False
        }

    return {
        "status": False,
        "message": red_fail("file was modified"),
        "current_hash": current_hash,
        "baseline_hash": baseline_hash,
        "baseline_created": False
    }

# ============================================================
# SCORE FROM METADATA JSON
# ============================================================
def load_eval_metadata(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def get_eval_scores(metadata):
    accuracy = metadata.get("accuracy")

    macro_f1 = metadata.get("macro_f1")

    classification_report = metadata.get("classification_report", {})

    if macro_f1 is None:
        macro_f1 = (
            classification_report
            .get("macro avg", {})
            .get("f1-score")
        )

    weighted_f1 = (
        classification_report
        .get("weighted avg", {})
        .get("f1-score")
    )

    macro_f1_score_out_of_25 = macro_f1 * MAX_SCORE if macro_f1 is not None else None

    return {
        "accuracy": accuracy,
        "f1_weighted": weighted_f1,
        "macro_f1": macro_f1,
        "macro_f1_score_out_of_25": macro_f1_score_out_of_25,
        "max_score": MAX_SCORE
    }
# ============================================================
# MAIN RUN
# ============================================================
def main():
    start_time = time.perf_counter()

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    integrity_result = check_file_not_modified(METADATA_JSON, BASELINE_HASH_FILE)

    if not integrity_result["status"]:
        print("\n********** File Integrity Check **********")
        print(f"Timestamp:          {now}")
        print(integrity_result["message"])
        print(f"Current Hash:       {integrity_result['current_hash']}")
        print(f"Baseline Hash:      {integrity_result['baseline_hash']}")
        print("******************************************")

        raise RuntimeError("Stop scoring because the metadata file was modified.")

    metadata = load_eval_metadata(METADATA_JSON)
    score_result = get_eval_scores(metadata)

    end_time = time.perf_counter()
    runtime_seconds = end_time - start_time

    print("\n********** Final Summary Score **********")
    print(f"Timestamp:              {now}")
    print(f"Runtime:                {runtime_seconds:.4f} seconds")
    print(f"File Integrity:          {integrity_result['message']}")
    print("----------------------------------------")
    print(f"Accuracy:               {score_result['accuracy']:.4f}")
    print(f"F1 Weighted:            {score_result['f1_weighted']:.4f}")
    print(f"Macro-F1:               {score_result['macro_f1']:.4f}")
    print("----------------------------------------")
    print(f"PHASE2: Macro-F1 Score / 25:    {score_result['macro_f1_score_out_of_25']:.4f}")
    print(f"Max Score:                      {score_result['max_score']}")
    print("****************************************")


if __name__ == "__main__":
    main()

# ***PHASE4: Calculate Unseen dataset***

In [ ]:
import csv
import json
import hashlib
import time
from pathlib import Path
from datetime import datetime

import pandas as pd

# ============================================================
# CONFIG
# ============================================================
PREDICT_FILE = r"" 

# ============================================================
GROUND_TRUTH_FILE = "plant_unseen_score_final.csv"
BASELINE_HASH_FILE = "file_integrity_baseline.json"
MAX_SCORE = 25

GT_REQUIRED_COLUMNS = [
    "timestamp",
    "temperature",
    "humidity",
    "lux",
    "vpd",
    "y",
    "y_label",
]

PRED_REQUIRED_COLUMNS = [
    "timestamp",
    "temperature",
    "humidity",
    "lux",
    "vpd",
    "y_pred",
    "y_label_pred",
    "y_pred_prob_normal",
    "y_pred_prob_alert",
    "y_pred_prob_alarm",
]

CLASS_ID_TO_LABEL = {
    0: "normal",
    1: "alert",
    2: "alarm",
}

PROB_COLUMNS = {
    0: "y_pred_prob_normal",
    1: "y_pred_prob_alert",
    2: "y_pred_prob_alarm",
}

# ============================================================
# COLOR PRINT
# ============================================================
GREEN = "\033[92m"
RED = "\033[91m"
YELLOW = "\033[93m"
RESET = "\033[0m"

def passed(msg):
    return f"{GREEN}PASSED{RESET} - {msg}"

def failed(msg):
    return f"{RED}FAILED{RESET} - {msg}"

def warning(msg):
    return f"{YELLOW}WARNING{RESET} - {msg}"

# ============================================================
# FILE STRUCTURE CHECK
# ============================================================
def check_csv_structure(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    bad_rows = []

    with open(file_path, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)

        header = next(reader)
        expected_cols = len(header)

        for line_no, row in enumerate(reader, start=2):
            if len(row) != expected_cols:
                bad_rows.append({
                    "line_no": line_no,
                    "expected_columns": expected_cols,
                    "actual_columns": len(row),
                    "row_preview": row[:15],
                })

    return {
        "status": len(bad_rows) == 0,
        "expected_columns": expected_cols,
        "bad_rows": bad_rows,
    }

# ============================================================
# FILE INTEGRITY CHECK
# ============================================================
def calculate_file_hash(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

def load_baseline_hashes(baseline_hash_file):
    baseline_hash_file = Path(baseline_hash_file)

    if not baseline_hash_file.exists():
        return {}

    with open(baseline_hash_file, "r", encoding="utf-8") as f:
        return json.load(f)

def save_baseline_hashes(baseline_hash_file, data):
    with open(baseline_hash_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

def check_file_not_modified(file_path, baseline_hash_file=BASELINE_HASH_FILE):
    file_path = Path(file_path)
    current_hash = calculate_file_hash(file_path)

    baseline_data = load_baseline_hashes(baseline_hash_file)

    file_key = file_path.name

    if file_key not in baseline_data:
        baseline_data[file_key] = {
            "baseline_hash": current_hash,
            "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

        save_baseline_hashes(baseline_hash_file, baseline_data)

        return {
            "status": True,
            "message": passed(f"{file_key} not modified, baseline hash created"),
            "file": file_key,
            "current_hash": current_hash,
            "baseline_hash": current_hash,
            "baseline_created": True,
        }

    baseline_hash = baseline_data[file_key]["baseline_hash"]

    if current_hash == baseline_hash:
        return {
            "status": True,
            "message": passed(f"{file_key} not modified"),
            "file": file_key,
            "current_hash": current_hash,
            "baseline_hash": baseline_hash,
            "baseline_created": False,
        }

    return {
        "status": False,
        "message": failed(f"{file_key} was modified"),
        "file": file_key,
        "current_hash": current_hash,
        "baseline_hash": baseline_hash,
        "baseline_created": False,
    }

# ============================================================
# DATA VALIDATION
# ============================================================
def check_required_columns(df, required_columns, file_name):
    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(
            f"{file_name} missing required columns: {missing}"
        )

    return True

def check_prediction_probability_matches_y_pred(pred_df):
    check_rows = []

    for idx, row in pred_df.iterrows():
        probs = {
            0: float(row["y_pred_prob_normal"]),
            1: float(row["y_pred_prob_alert"]),
            2: float(row["y_pred_prob_alarm"]),
        }

        expected_y_pred = max(probs, key=probs.get)
        actual_y_pred = int(float(row["y_pred"]))

        expected_label = CLASS_ID_TO_LABEL[expected_y_pred]
        actual_label = str(row["y_label_pred"]).strip().lower()

        prob_match_y_pred = expected_y_pred == actual_y_pred
        label_match_y_pred = expected_label == actual_label

        if not prob_match_y_pred or not label_match_y_pred:
            check_rows.append({
                "row_index": idx,
                "timestamp": row.get("timestamp"),
                "actual_y_pred": actual_y_pred,
                "expected_y_pred_from_prob": expected_y_pred,
                "actual_y_label_pred": actual_label,
                "expected_label_from_prob": expected_label,
                "prob_normal": probs[0],
                "prob_alert": probs[1],
                "prob_alarm": probs[2],
                "prob_match_y_pred": prob_match_y_pred,
                "label_match_y_pred": label_match_y_pred,
            })

    return {
        "status": len(check_rows) == 0,
        "total_rows": len(pred_df),
        "failed_rows": len(check_rows),
        "details": pd.DataFrame(check_rows),
    }

# ============================================================
# CLASSIFICATION METRICS
# ============================================================
def calculate_classification_metrics(y_true, y_pred, labels=(0, 1, 2)):
    y_true = [int(float(v)) for v in y_true]
    y_pred = [int(float(v)) for v in y_pred]

    total = len(y_true)
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    accuracy = correct / total if total > 0 else 0

    class_rows = []

    for label in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == label and p == label)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != label and p == label)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == label and p != label)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0
        )

        support = sum(1 for t in y_true if t == label)

        class_rows.append({
            "class_id": label,
            "class_label": CLASS_ID_TO_LABEL[label],
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
            "tp": tp,
            "fp": fp,
            "fn": fn,
        })

    class_metric_df = pd.DataFrame(class_rows)

    macro_precision = class_metric_df["precision"].mean()
    macro_recall = class_metric_df["recall"].mean()
    macro_f1 = class_metric_df["f1"].mean()

    macro_f1_score_out_of_25 = macro_f1 * MAX_SCORE

    return {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "macro_f1_score_out_of_25": macro_f1_score_out_of_25,
        "max_score": MAX_SCORE,
        "class_metric_df": class_metric_df,
    }

# ============================================================
# COMPARE WITH GROUND TRUTH
# ============================================================
def compare_with_ground_truth(gt_df, pred_df):
    gt = gt_df.copy()
    pred = pred_df.copy()

    gt["timestamp"] = pd.to_datetime(gt["timestamp"])
    pred["timestamp"] = pd.to_datetime(pred["timestamp"])

    gt["y"] = gt["y"].astype(float).astype(int)
    pred["y_pred"] = pred["y_pred"].astype(float).astype(int)

    gt["y_label"] = gt["y_label"].astype(str).str.strip().str.lower()
    pred["y_label_pred"] = pred["y_label_pred"].astype(str).str.strip().str.lower()

    merged = gt.merge(
        pred,
        on="timestamp",
        how="inner",
        suffixes=("_gt", "_pred"),
    )

    missing_prediction_rows = len(gt) - len(merged)

    if len(merged) == 0:
        raise ValueError("No matching timestamp rows between ground truth and prediction file.")

    metrics = calculate_classification_metrics(
        y_true=merged["y"],
        y_pred=merged["y_pred"],
        labels=(0, 1, 2),
    )

    merged["y_match"] = merged["y"] == merged["y_pred"]
    merged["y_label_match"] = merged["y_label"] == merged["y_label_pred"]

    label_match_accuracy = merged["y_label_match"].mean()

    return {
        "merged_df": merged,
        "matched_timestamp_rows": len(merged),
        "ground_truth_rows": len(gt),
        "prediction_rows": len(pred),
        "missing_prediction_rows": missing_prediction_rows,
        "label_match_accuracy": label_match_accuracy,
        **metrics,
    }

# ============================================================
# MAIN
# ============================================================
def main():
    start_time = time.perf_counter()
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    pred_structure = check_csv_structure(PREDICT_FILE)
    gt_structure = check_csv_structure(GROUND_TRUTH_FILE)

    if not pred_structure["status"]:
        print("\n********** CSV Structure Check **********")
        print(f"Timestamp:          {now}")
        print(f"Prediction File:    {failed('bad CSV structure')}")
        print(f"Expected Columns:   {pred_structure['expected_columns']}")
        print(f"Bad Row Count:      {len(pred_structure['bad_rows'])}")

        print("\nFirst 5 bad rows:")
        for bad in pred_structure["bad_rows"][:5]:
            print(
                f"Line {bad['line_no']}: "
                f"expected {bad['expected_columns']}, "
                f"actual {bad['actual_columns']}, "
                f"preview={bad['row_preview']}"
            )

        print("*****************************************")
        raise RuntimeError("Stop scoring because prediction CSV structure is invalid.")

    if not gt_structure["status"]:
        print("\n********** CSV Structure Check **********")
        print(f"Timestamp:          {now}")
        print(f"Ground Truth File:  {failed('bad CSV structure')}")
        print(f"Expected Columns:   {gt_structure['expected_columns']}")
        print(f"Bad Row Count:      {len(gt_structure['bad_rows'])}")
        print("*****************************************")
        raise RuntimeError("Stop scoring because ground-truth CSV structure is invalid.")

    pred_integrity = check_file_not_modified(PREDICT_FILE)
    gt_integrity = check_file_not_modified(GROUND_TRUTH_FILE)

    if not pred_integrity["status"] or not gt_integrity["status"]:
        print("\n********** File Integrity Check **********")
        print(f"Timestamp:          {now}")
        print(pred_integrity["message"])
        print(gt_integrity["message"])
        print("******************************************")
        raise RuntimeError("Stop scoring because one or more files were modified.")

    gt_df = pd.read_csv(GROUND_TRUTH_FILE)
    pred_df = pd.read_csv(PREDICT_FILE)

    check_required_columns(gt_df, GT_REQUIRED_COLUMNS, GROUND_TRUTH_FILE)
    check_required_columns(pred_df, PRED_REQUIRED_COLUMNS, PREDICT_FILE)

    prob_check = check_prediction_probability_matches_y_pred(pred_df)

    if not prob_check["status"]:
        prob_check["details"].to_csv(
            "prediction_probability_mismatch_rows.csv",
            index=False
        )

        print("\n********** Prediction Probability Check **********")
        print(f"Timestamp:          {now}")
        print(f"Status:             {failed('probability columns do not match y_pred')}")
        print(f"Total Rows:         {prob_check['total_rows']}")
        print(f"Failed Rows:        {prob_check['failed_rows']}")
        print("Mismatch File:      prediction_probability_mismatch_rows.csv")
        print("************************************************")
        raise RuntimeError("Stop scoring because y_pred does not match probability argmax.")

    result = compare_with_ground_truth(gt_df, pred_df)

    result["merged_df"].to_csv("ground_truth_prediction_comparison.csv", index=False)
    result["class_metric_df"].to_csv("classification_metrics_by_class.csv", index=False)

    end_time = time.perf_counter()
    runtime_seconds = end_time - start_time

    print("\n********** Final Summary Score **********")
    print(f"Timestamp:                  {now}")
    print(f"Runtime:                    {runtime_seconds:.4f} seconds")
    print("----------------------------------------")
    print(f"Prediction File Integrity:  {pred_integrity['message']}")
    print(f"Ground Truth Integrity:     {gt_integrity['message']}")
    print(f"CSV Structure Check:        {passed('valid CSV structure')}")
    print(f"Probability Check:          {passed('all probability rows match y_pred')}")
    print("----------------------------------------")
    print(f"Ground Truth Rows:          {result['ground_truth_rows']}")
    print(f"Prediction Rows:            {result['prediction_rows']}")
    print(f"Matched Timestamp Rows:     {result['matched_timestamp_rows']}")
    print(f"Missing Prediction Rows:    {result['missing_prediction_rows']}")
    print("----------------------------------------")
    print(f"Accuracy:                   {result['accuracy']:.4f}")
    print(f"Macro Precision:            {result['macro_precision']:.4f}")
    print(f"Macro Recall:               {result['macro_recall']:.4f}")
    print(f"Macro-F1:                   {result['macro_f1']:.4f}")
    print(f"Label Match Accuracy:       {result['label_match_accuracy']:.4f}")
    print("----------------------------------------")
    print(f"PHASE4: Final Score / 25:   {result['macro_f1_score_out_of_25']:.4f}")
    print(f"Max Score:                  {result['max_score']}")
    print("----------------------------------------")
    print("Saved Comparison File:      ground_truth_prediction_comparison.csv")
    print("Saved Class Metrics File:   classification_metrics_by_class.csv")
    print("****************************************")

if __name__ == "__main__":
    main()